In [ ]:
# 🌐 Setup ngrok tunnel for API access
print("🌐 Setting up ngrok tunnel...")

# Check if repository exists
from pathlib import Path
import subprocess
import time

ROOT = Path('/content/egyption_id_ready')
if not ROOT.exists():
    print("❌ Error: Repository not found!")
    print("⚠️  You must run Part 1 (Environment Setup) first!")
    print("\n📋 Quick fix:")
    print("   1. Scroll to Part 1")
    print("   2. Run all cells from Part 1 to Part 3")
    print("   3. Then run this cell again")
else:
    print(f"✅ Repository found: {ROOT}")
    
    # Install pyngrok
    print("\n📦 Installing pyngrok...")
    !pip install -q pyngrok
    
    # Set ngrok auth token (optional but recommended)
    NGROK_AUTH_TOKEN = ""  # ← Add your token from https://dashboard.ngrok.com
    
    from pyngrok import ngrok, conf
    
    if NGROK_AUTH_TOKEN:
        conf.get_default().auth_token = NGROK_AUTH_TOKEN
        print("✅ Using authenticated ngrok tunnel")
    else:
        print("⚠️  Using anonymous tunnel (limited to 1 connection)")
        print("   Get free auth token: https://dashboard.ngrok.com/get-started/your-authtoken")
    
    # Kill any existing tunnels
    ngrok.kill()
    
    # Start API server
    print("\n🚀 Starting FastAPI server...")
    print(f"   Working directory: {ROOT}")
    
    try:
        api_process = subprocess.Popen(
            ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
            cwd=str(ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )
        
        # Wait for server to start
        time.sleep(5)
        
        # Check if process is still running
        if api_process.poll() is None:
            # Create tunnel
            public_url = ngrok.connect(8000)
            print(f"\n✅ API server started!")
            print(f"🔗 Public API URL: {public_url}")
            print(f"📖 API Docs: {public_url}/docs")
            print(f"📖 API Health: {public_url}/")
            print(f"\n💡 Keep this cell running to maintain the tunnel")
            print(f"   To stop: press the stop button or run: ngrok.kill()")
        else:
            stdout, stderr = api_process.communicate()
            print(f"\n❌ API server failed to start!")
            print(f"   Error: {stderr.decode()}")
            print(f"\n⚠️  Make sure you ran all previous cells first!")
            
    except FileNotFoundError as e:
        print(f"\n❌ Error: {e}")
        print(f"\n⚠️  The repository directory doesn't exist!")
        print(f"   Please run Part 1 (Environment Setup) first!")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")

### 📊 Dataset Information (from thinkai-engine)

**Egyptian ID OCR Dataset - Complete**

| Statistic | Value |
|-----------|-------|
| **Total Images** | 16,720 ID cards |
| **Training** | 15,669 images |
| **Validation** | 948 images |
| **Test** | 103 images |
| **Field Crops** | ~57,685 fields |
| **Supported Fields** | 24 fields |
| **Format** | YOLO labels |
| **Size** | ~2.5 GB |

**Download Source:**
- Primary: Git LFS (thinkai-engine/egyption_id_ready)
- Backup: GitHub Releases

**Repository:** https://github.com/thinkai-engine/egyption_id_ready

**License:** Open for research and educational use

In [ ]:
# 📊 Download Egyptian ID Dataset from GitHub (thinkai-engine)
# Using your repository - no Google Drive needed!

from pathlib import Path
import subprocess
import os

ROOT = Path('/content/egyption_id_ready')

print("📥 Downloading Egyptian ID OCR Dataset...")
print("   Source: GitHub (thinkai-engine/egyption_id_ready)")
print("   Method: Git LFS + Direct Downloads")
print("   Size: ~2.5 GB")
print("   Images: 16,720 ID cards")
print()

# Install Git LFS for large file downloads
print("📦 Installing Git LFS...")
!apt-get install -y git-lfs
!git lfs install

# Navigate to repo and pull LFS files
print("\n📥 Downloading dataset with Git LFS...")
%cd /content/egyption_id_ready
!git lfs pull

# Verify dataset
print("\n🔍 Verifying dataset...")
for split in ['train', 'valid', 'test']:
    split_path = ROOT / split / 'images'
    if split_path.exists():
        img_count = len(list(split_path.glob('*.jpg')))
        lbl_count = len(list((ROOT / split / 'labels').glob('*.txt')))
        print(f"   ✅ {split}: {img_count:,} images, {lbl_count:,} labels")
    else:
        print(f"   ⚠️  {split}: Not found")

%cd /content/egyption_id_ready

print("\n✅ Dataset downloaded successfully!")

In [ ]:
# 📊 Download Egyptian ID Dataset from HuggingFace
# Using public dataset - no Google Drive needed!

from pathlib import Path
ROOT = Path('/content/egyption_id_ready')

print("📥 Downloading Egyptian ID OCR Dataset...")
print("   Source: HuggingFace Datasets")
print("   Size: ~2.5 GB")
print("   Images: 16,720 ID cards")
print()

# Install HuggingFace datasets library
!pip install -q datasets

# Download dataset
from datasets import load_dataset

print("⏳ Loading dataset (this may take 5-10 minutes)...")
try:
    # Try to load from HuggingFace
    dataset = load_dataset("NAMO7Y/Egyptian_ID_OCR_Dataset")
    
    # Save to disk
    print("\n💾 Saving to disk...")
    for split in ['train', 'validation', 'test']:
        if split in dataset:
            split_dir = ROOT / split
            split_dir.mkdir(parents=True, exist_ok=True)
            
            # Save images and labels
            print(f"   Processing {split} split...")
            dataset[split].to_csv(split_dir / "metadata.csv")
    
    print("\n✅ Dataset downloaded successfully!")
    
except Exception as e:
    print(f"⚠️  HuggingFace download failed: {e}")
    print("\n📂 Falling back to GitHub release download...")
    
    # Alternative: Download from GitHub Releases
    DATASET_URL = "https://github.com/NAMO7Y/Egyptian_ID_OCR/releases/download/v1.0.0/dataset.zip"
    !wget -q --show-progress -O {ROOT}/dataset.zip {DATASET_URL}
    
    # Extract
    print("\n📦 Extracting dataset...")
    !unzip -q {ROOT}/dataset.zip -d {ROOT}
    !rm {ROOT}/dataset.zip
    
    print("\n✅ Dataset downloaded from GitHub!")

In [ ]:
# 📊 Track Your Progress
# Run this cell anytime to see where you are

from pathlib import Path
import pandas as pd

ROOT = Path('/content/egyption_id_ready')

print("=" * 60)
print("📊 PROGRESS TRACKER")
print("=" * 60)
print()

# Part 1: Setup
if ROOT.exists():
    print("✅ Part 1: Environment Setup - COMPLETE")
else:
    print("⏳ Part 1: Environment Setup - PENDING")

# Part 2: Models
if (ROOT / 'weights' / 'card_detection.pt').exists():
    print("✅ Part 2: Download Models - COMPLETE")
else:
    print("⏳ Part 2: Download Models - PENDING")

# Part 3: Dataset
if (ROOT / 'train' / 'images').exists():
    img_count = len(list((ROOT / 'train' / 'images').glob('*.jpg')))
    print(f"✅ Part 3: Download Dataset - COMPLETE ({img_count} images)")
else:
    print("⏳ Part 3: Download Dataset - PENDING")

# Part 4: Processing
if (ROOT / 'rec' / 'images' / 'two_stage').exists():
    crop_count = len(list((ROOT / 'rec' / 'images' / 'two_stage').glob('*.jpg')))
    print(f"✅ Part 4: Build Dataset - COMPLETE ({crop_count:,} crops)")
else:
    print("⏳ Part 4: Build Dataset - PENDING")

# Part 5: Labeling
if (ROOT / 'crops_labeled.csv').exists():
    df = pd.read_csv(ROOT / 'crops_labeled.csv')
    print(f"✅ Part 5: Label Crops - COMPLETE ({len(df):,} labeled)")
else:
    print("⏳ Part 5: Label Crops - PENDING")

# Part 6: Training Data
if (ROOT / 'rec' / 'train.txt').exists():
    with open(ROOT / 'rec' / 'train.txt') as f:
        lines = len(f.readlines())
    print(f"✅ Part 6: Prepare Training Data - COMPLETE ({lines} samples)")
else:
    print("⏳ Part 6: Prepare Training Data - PENDING")

# Part 7: Training
if (ROOT / 'output' / 'egyptian_id_rec').exists():
    checkpoints = list((ROOT / 'output' / 'egyptian_id_rec').glob('*.pdparams'))
    print(f"✅ Part 7: Training - {'IN PROGRESS' if checkpoints else 'PENDING'}")
else:
    print("⏳ Part 7: Training - PENDING")

# Part 8-11: Evaluation, Export, API
if (ROOT / 'onnx' / 'rec_sim.onnx').exists():
    size_mb = (ROOT / 'onnx' / 'rec_sim.onnx').stat().st_size / 1024 / 1024
    print(f"✅ Parts 8-11: Export & Deploy - COMPLETE ({size_mb:.1f} MB)")
else:
    print("⏳ Parts 8-11: Export & Deploy - PENDING")

print()
print("=" * 60)

In [ ]:
# 📊 Alternative Dataset Sources
# If the main dataset download fails, try these alternatives

from pathlib import Path
ROOT = Path('/content/egyption_id_ready')

print("📋 Alternative Dataset Sources:")
print()
print("1️⃣ HuggingFace Datasets (Recommended):")
print("   !pip install datasets")
print("   from datasets import load_dataset")
print("   dataset = load_dataset('NAMO7Y/Egyptian_ID_OCR_Dataset')")
print()
print("2️⃣ GitHub Releases:")
print("   !wget https://github.com/NAMO7Y/Egyptian_ID_OCR/releases/download/v1.0.0/dataset.zip")
print()
print("3️⃣ Google Drive (Manual):")
print("   Upload your dataset.zip to Google Drive")
print("   Use: from google.colab import drive")
print("        drive.mount('/content/drive')")
print()
print("4️⃣ Create Your Own:")
print("   Upload images to /content/egyption_id_ready/train/images/")
print("   Upload labels to /content/egyption_id_ready/train/labels/")
print("   Run: python scripts/process_full_dataset_two_stage.py")

print("\n💡 Tip: Start with sample dataset for testing!")

In [ ]:
# Check GPU
!nvidia-smi

# Check Python version
!python --version

# Mount Google Drive (optional - for saving outputs)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

print("✅ Environment check complete!")

### 1.2 Clone Repository and Install Dependencies

In [ ]:
# Clone the repository
%cd /content
!git clone https://github.com/thinkai-engine/egyption_id_ready.git
%cd /content/egyption_id_ready

# Install system dependencies
!apt-get update && apt-get install -y libgl1-mesa-glx libglib2.0-0 libgomp1 curl wget git

print("✅ Repository cloned and system dependencies installed!")

In [ ]:
# Clone PaddleOCR repository (required for training)
%cd /content
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
print("✅ PaddleOCR cloned!")

In [ ]:
# Install Python dependencies
!pip install -q -r requirements.txt
!pip install -q google-generativeai

print("✅ Python dependencies installed!")

In [ ]:
# Helper function for robust downloads
def download_with_retry(url, output_path, max_retries=3, timeout=60):
    """Download file with retry logic."""
    import subprocess
    from pathlib import Path
    
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    
    for attempt in range(max_retries):
        try:
            result = subprocess.run(
                ['wget', '-q', '--timeout=' + str(timeout), 
                 '--tries=1', '-O', str(output), url],
                capture_output=True,
                text=True,
                timeout=timeout * 2
            )
            if result.returncode == 0 and output.exists() and output.stat().st_size > 0:
                return True
        except Exception as e:
            print(f"Attempt {attempt + 1}/{max_retries} failed: {e}")
    
    return False

print("✅ Download helper loaded!")

### 1.3 Verify Installation

In [ ]:
import torch, cv2, pandas as pd, numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"OpenCV: {cv2.__version__}")
print(f"Pandas: {pd.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n✅ Installation verified!")

## 📥 Part 2: Download Models

### 2.1 Download YOLO Detection Models from GitHub

In [ ]:
!mkdir -p /content/egyption_id_ready/weights

print("⬇️ Downloading Card Detection Model...")
!wget -q --show-progress -O /content/egyption_id_ready/weights/card_detection.pt https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/detect_id_card.pt

print("\n⬇️ Downloading Field Detection Model...")
!wget -q --show-progress -O /content/egyption_id_ready/weights/field_detection.pt https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/detect_odjects.pt

print("\n⬇️ Downloading NID Detection Model...")
!wget -q --show-progress -O /content/egyption_id_ready/weights/nid_detection.pt https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/detect_id.pt

print("\n📦 Model Files:")
!ls -lh /content/egyption_id_ready/weights/

In [ ]:
# Download Arabic dictionary file (required for OCR)
!wget -q -O /content/egyption_id_ready/arabic_dict.txt \
    https://raw.githubusercontent.com/NAMO7Y/Egyptian_ID_OCR/main/arabic_dict.txt

# Verify file exists
from pathlib import Path
dict_path = Path('/content/egyption_id_ready/arabic_dict.txt')
if dict_path.exists():
    print(f"✅ Arabic dictionary downloaded: {dict_path.stat().st_size} bytes")
else:
    print("❌ Arabic dictionary download failed!")
    # Create minimal dictionary as fallback
    with open(dict_path, 'w', encoding='utf-8') as f:
        # Write basic Arabic characters
        arabic_chars = ''.join([chr(i) for i in range(0x0600, 0x06FF)])
        f.write(arabic_chars)
    print("⚠️  Created minimal fallback dictionary")

In [ ]:
# Download field detector ONNX model (required for inference)
!mkdir -p /content/egyption_id_ready/model

# Try to download from project repository
# Note: Replace with actual model URL if different
!wget -q --timeout=30 --tries=3 -O /content/egyption_id_ready/model/field_detector.onnx \
    https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/field_detector.onnx || \
    echo "⚠️  Field detector model not available - will use YOLO models instead"

# Verify download
from pathlib import Path
model_path = Path('/content/egyption_id_ready/model/field_detector.onnx')
if model_path.exists() and model_path.stat().st_size > 0:
    print(f"✅ Field detector model downloaded: {model_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print("⚠️  Field detector model not available - two-stage detection will use YOLO models only")

In [ ]:
# Validate all required models are downloaded
from pathlib import Path

ROOT = Path('/content/egyption_id_ready')
required_files = {
    'weights/card_detection.pt': 'Card Detection Model',
    'weights/field_detection.pt': 'Field Detection Model',
    'weights/nid_detection.pt': 'NID Detection Model',
    'arabic_PP-OCRv3_rec/best_accuracy.pdparams': 'PaddleOCR Model',
    'arabic_dict.txt': 'Arabic Dictionary',
}

print("🔍 Validating downloaded files...")
all_ok = True
for file_path, description in required_files.items():
    full_path = ROOT / file_path
    if full_path.exists() and full_path.stat().st_size > 0:
        size_mb = full_path.stat().st_size / 1024 / 1024
        print(f"  ✅ {description}: {size_mb:.2f} MB")
    else:
        print(f"  ❌ {description}: MISSING")
        all_ok = False

if all_ok:
    print("\n✅ All models downloaded successfully!")
else:
    print("\n⚠️  Some models are missing. Check error messages above.")

In [ ]:
# Monitor disk space
!df -h /content

# Clean unnecessary files to save space
print("\n🧹 Cleaning unnecessary files...")
!rm -rf /content/egyption_id_ready/.git 2>/dev/null || true
!rm -rf /content/PaddleOCR/.git 2>/dev/null || true
!rm -rf /content/egyption_id_ready/__pycache__ 2>/dev/null || true
!rm -rf /content/egyption_id_ready/src/__pycache__ 2>/dev/null || true
print("✅ Cleanup complete!")

# Show freed space
!df -h /content

### 2.2 Download OCR Models

In [ ]:
print("⬇️ Preparing EasyOCR models (Arabic + English)...")
!mkdir -p /content/egyption_id_ready/models_cache

import easyocr
reader = easyocr.Reader(['ar', 'en'], gpu=True, model_storage_directory='/content/egyption_id_ready/models_cache')
print("✅ EasyOCR initialized!")

print("\n⬇️ Downloading PaddleOCR pretrained model...")
!mkdir -p /content/egyption_id_ready/arabic_PP-OCRv3_rec
!wget -q --show-progress -O /content/egyption_id_ready/arabic_PP-OCRv3_rec/best_accuracy.pdparams https://paddleocr.bj.bcebos.com/PP-OCRv3/arabic/rec_arabic_ppocr_v3_train/best_accuracy.pdparams

print("\n✅ OCR models ready!")

In [ ]:
# 📦 Download ALL Files from thinkai-engine/egyption_id_ready
# This downloads everything needed from your GitHub repository

from pathlib import Path
import subprocess

ROOT = Path('/content/egyption_id_ready')

print("=" * 60)
print("📦 DOWNLOADING ALL FILES FROM GITHUB")
print("   Repository: thinkai-engine/egyption_id_ready")
print("=" * 60)
print()

# Install Git LFS for large files
print("📦 Installing Git LFS...")
!apt-get install -y git-lfs > /dev/null 2>&1
!git lfs install > /dev/null 2>&1

# Navigate to repo
%cd /content/egyption_id_ready

# Pull all LFS files
print("\n📥 Pulling Git LFS files (this may take 5-10 minutes)...")
!git lfs pull

# Download additional models from releases if available
print("\n📥 Checking for additional models in releases...")

# Create downloads directory
downloads_dir = ROOT / 'downloads'
downloads_dir.mkdir(exist_ok=True)

# Try to download from GitHub Releases
RELEASE_URL = "https://github.com/thinkai-engine/egyption_id_ready/releases/latest/download/models.zip"
try:
    result = subprocess.run(
        ['wget', '-q', '--show-progress', '--timeout=60', '--tries=3',
         '-O', str(downloads_dir / 'models.zip'), RELEASE_URL],
        capture_output=True,
        timeout=300
    )
    if result.returncode == 0:
        print("✅ Models downloaded from releases")
        # Extract
        !unzip -q {downloads_dir}/models.zip -d {ROOT}
        !rm {downloads_dir}/models.zip
    else:
        print("⚠️  No models.zip in releases - using LFS files")
except Exception as e:
    print(f"⚠️  Release download failed: {e}")

# Validate all downloads
print("\n" + "=" * 60)
print("🔍 VALIDATING ALL DOWNLOADS")
print("=" * 60)

required_files = {
    # Dataset
    'train/images': 'Training Images',
    'train/labels': 'Training Labels',
    'valid/images': 'Validation Images',
    'valid/labels': 'Validation Labels',
    'test/images': 'Test Images',
    'test/labels': 'Test Labels',
    
    # Models
    'weights/card_detection.pt': 'Card Detection Model',
    'weights/field_detection.pt': 'Field Detection Model',
    'model/field_detector.onnx': 'Field Detector ONNX',
    'arabic_dict.txt': 'Arabic Dictionary',
    
    # Code
    'src/inference.py': 'Inference Module',
    'app/main.py': 'API Server',
    'configs/egyptian_id_rec.yml': 'Training Config',
}

all_ok = True
for file_path, description in required_files.items():
    full_path = ROOT / file_path
    if full_path.exists():
        if full_path.is_dir():
            count = len(list(full_path.glob('*')))
            print(f"  ✅ {description}: {count} files")
        else:
            size_mb = full_path.stat().st_size / 1024 / 1024
            print(f"  ✅ {description}: {size_mb:.2f} MB")
    else:
        print(f"  ❌ {description}: MISSING")
        all_ok = False

if all_ok:
    print("\n✅ ALL FILES DOWNLOADED SUCCESSFULLY!")
    print("\n📂 Repository Structure:")
    print(f"   Root: {ROOT}")
    print(f"   Dataset: {ROOT / 'train'} / {ROOT / 'valid'} / {ROOT / 'test'}")
    print(f"   Models: {ROOT / 'weights'} / {ROOT / 'model'}")
    print(f"   Code: {ROOT / 'src'} / {ROOT / 'app'}")
    print(f"   Configs: {ROOT / 'configs'}")
else:
    print("\n⚠️  Some files are missing. Check messages above.")

%cd /content/egyption_id_ready

In [ ]:
# 📦 Complete Data Download - All Files from GitHub
# This downloads everything needed from GitHub repositories

from pathlib import Path
import subprocess

ROOT = Path('/content/egyption_id_ready')

print("=" * 60)
print("📦 DOWNLOADING ALL REQUIRED FILES FROM GITHUB")
print("=" * 60)

# Define all downloads
downloads = {
    # YOLO Models (Naso7y GitHub)
    'weights/card_detection.pt': 
        'https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/detect_id_card.pt',
    'weights/field_detection.pt': 
        'https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/detect_odjects.pt',
    'weights/nid_detection.pt': 
        'https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/detect_id.pt',
    
    # PaddleOCR Model
    'arabic_PP-OCRv3_rec/best_accuracy.pdparams': 
        'https://paddleocr.bj.bcebos.com/PP-OCRv3/arabic/rec_arabic_ppocr_v3_train/best_accuracy.pdparams',
    
    # Arabic Dictionary
    'arabic_dict.txt': 
        'https://raw.githubusercontent.com/NAMO7Y/Egyptian_ID_OCR/main/arabic_dict.txt',
    
    # Field Detector ONNX
    'model/field_detector.onnx': 
        'https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/field_detector.onnx',
}

# Download each file
for file_path, url in downloads.items():
    full_path = ROOT / file_path
    full_path.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"\n⬇️  Downloading: {file_path}")
    print(f"   URL: {url[:80]}...")
    
    try:
        result = subprocess.run(
            ['wget', '-q', '--show-progress', '--timeout=60', '--tries=3', 
             '-O', str(full_path), url],
            capture_output=True,
            text=True,
            timeout=300
        )
        
        if result.returncode == 0 and full_path.exists() and full_path.stat().st_size > 0:
            size_mb = full_path.stat().st_size / 1024 / 1024
            print(f"   ✅ Success: {size_mb:.2f} MB")
        else:
            print(f"   ⚠️  Downloaded but may be incomplete")
            
    except subprocess.TimeoutExpired:
        print(f"   ⚠️  Download timed out (trying next file)")
    except Exception as e:
        print(f"   ❌ Failed: {e}")

# Final validation
print("\n" + "=" * 60)
print("🔍 VALIDATING DOWNLOADS")
print("=" * 60)

all_ok = True
for file_path, _ in downloads.items():
    full_path = ROOT / file_path
    if full_path.exists() and full_path.stat().st_size > 0:
        size_mb = full_path.stat().st_size / 1024 / 1024
        print(f"  ✅ {file_path}: {size_mb:.2f} MB")
    else:
        print(f"  ❌ {file_path}: MISSING")
        all_ok = False

if all_ok:
    print("\n✅ ALL FILES DOWNLOADED SUCCESSFULLY!")
else:
    print("\n⚠️  Some files failed to download. Check messages above.")

## 📊 Part 3: Download Dataset

### 3.1 Download Egyptian ID Dataset from Google Drive

**Dataset Link**: Replace with your Google Drive link

The dataset should contain:
- `train/images/` - Training images + labels
- `valid/images/` - Validation images + labels  
- `test/images/` - Test images + labels

In [ ]:
FILE_ID = "YOUR_DATASET_FILE_ID_HERE"
OUTPUT_PATH = "/content/egyption_id_ready/dataset.zip"

print(f"⬇️ Downloading dataset from Google Drive...")
!pip install -q gdown
!gdown --id {FILE_ID} -O {OUTPUT_PATH}

print("\n✅ Dataset downloaded!")

In [ ]:
print("📦 Extracting dataset...")
!unzip -q /content/egyption_id_ready/dataset.zip -d /content/egyption_id_ready/

print("\n📊 Dataset Statistics:")
for split in ['train', 'valid', 'test']:
    img_count = !ls /content/egyption_id_ready/{split}/images/*.jpg 2>/dev/null | wc -l
    lbl_count = !ls /content/egyption_id_ready/{split}/labels/*.txt 2>/dev/null | wc -l
    print(f"   {split:6s}: {img_count[0]:>6} images, {lbl_count[0]:>6} labels")

print("\n✅ Dataset ready!")

## 🔧 Part 4: Build Dataset with Two-Stage Detection

In [ ]:
import sys
from pathlib import Path
ROOT = Path('/content/egyption_id_ready')
sys.path.insert(0, str(ROOT))
print(f"📂 Project root: {ROOT}")

In [ ]:
import cv2, matplotlib.pyplot as plt
from src.card_detector import load_card_detector

print("🔧 Loading two-stage detector...")
detector = load_card_detector(
    card_model_path=str(ROOT / 'weights' / 'card_detection.pt'),
    field_model_path=str(ROOT / 'weights' / 'field_detection.pt'),
)
print("✅ Detector loaded!")

test_images = list((ROOT / 'train' / 'images').glob('*.jpg'))
if test_images:
    image = cv2.imread(str(test_images[0]))
    card_crop, fields = detector.detect_full(image, translate_to_project=True)
    print(f"✂️  Card crop: {card_crop.shape[1]}x{card_crop.shape[0]}")
    print(f"📋 Fields detected: {len(fields)}")

In [ ]:
print("🚀 Processing full dataset...")
!python /content/egyption_id_ready/scripts/process_full_dataset_two_stage.py --splits train valid test
print("\n✅ Dataset processing complete!")
!ls -lh /content/egyption_id_ready/rec/images/two_stage/ | head -5

In [ ]:
# Validate dataset processing results
from pathlib import Path
import pandas as pd

print("🔍 Validating dataset processing...")

# Check cropped images
crops_dir = Path('/content/egyption_id_ready/rec/images/two_stage')
if crops_dir.exists():
    crop_count = len(list(crops_dir.glob('*.jpg')))
    print(f"  ✅ Cropped fields: {crop_count:,} images")
else:
    print("  ❌ Crops directory not found!")

# Check metadata CSV
metadata_path = Path('/content/egyption_id_ready/crops_metadata_two_stage.csv')
if metadata_path.exists():
    df = pd.read_csv(metadata_path)
    print(f"  ✅ Metadata: {len(df):,} records")
    print(f"     Fields: {df['field'].nunique()} unique")
    print(f"     Avg confidence: {df['confidence'].mean():.3f}")
else:
    print("  ❌ Metadata CSV not found!")

print("\n✅ Dataset validation complete!")

## 🏷️ Part 5: Label Crops with OCR

In [ ]:
OCR_METHOD = "qari-airllm"
print(f"🏷️ Labeling crops with {OCR_METHOD}...")

!python /content/egyption_id_ready/scripts/label_crops.py --method {OCR_METHOD} --splits train valid

print("\n✅ Labeling complete!")
!wc -l /content/egyption_id_ready/crops_labeled.csv

## 🎯 Part 6: Prepare Training Data

In [ ]:
print("📝 Preparing training data...")
!python /content/egyption_id_ready/scripts/prepare_paddle_labels.py --input /content/egyption_id_ready/crops_labeled.csv --output-dir /content/egyption_id_ready/rec
print("\n✅ Training data prepared!")
!head -3 /content/egyption_id_ready/rec/train.txt

## 🚀 Part 7: Train PaddleOCR Model

In [ ]:
# ✅ Prerequisite Check Before Training
# Run this cell to verify everything is ready

from pathlib import Path

ROOT = Path('/content/egyption_id_ready')

print("🔍 Checking training prerequisites...")
print()

checks = {
    'Repository': ROOT.exists(),
    'Weights': (ROOT / 'weights').exists(),
    'Card Model': (ROOT / 'weights' / 'card_detection.pt').exists(),
    'Field Model': (ROOT / 'weights' / 'field_detection.pt').exists(),
    'PaddleOCR Model': (ROOT / 'arabic_PP-OCRv3_rec' / 'best_accuracy.pdparams').exists(),
    'Arabic Dict': (ROOT / 'arabic_dict.txt').exists(),
    'Train Data': (ROOT / 'rec' / 'train.txt').exists(),
    'Config': (ROOT / 'configs' / 'egyptian_id_rec.yml').exists(),
    'PaddleOCR': (ROOT / 'PaddleOCR').exists(),
}

all_ok = True
for name, passed in checks.items():
    status = "✅" if passed else "❌"
    print(f"  {status} {name}")
    if not passed:
        all_ok = False

print()
if all_ok:
    print("✅ All prerequisites met! Ready to train!")
    print()
    print("🚀 You can now run the training cell")
else:
    print("❌ Some prerequisites are missing!")
    print()
    print("📋 Run these parts first:")
    if not checks['Repository']:
        print("   - Part 1: Environment Setup")
    if not checks['Weights'] or not checks['Card Model'] or not checks['Field Model']:
        print("   - Part 2: Download Models")
    if not checks['PaddleOCR Model'] or not checks['Arabic Dict']:
        print("   - Part 2: Download Models (continued)")
    if not checks['Train Data']:
        print("   - Part 4: Build Dataset")
        print("   - Part 5: Label Crops")
        print("   - Part 6: Prepare Training Data")
    if not checks['PaddleOCR']:
        print("   - Clone PaddleOCR repository")

In [ ]:
# Check for existing checkpoints before training
from pathlib import Path
checkpoint_dir = Path('/content/egyption_id_ready/output/egyptian_id_rec')
latest_checkpoint = checkpoint_dir / 'latest.pdparams'

if latest_checkpoint.exists():
    print(f"📋 Found existing checkpoint: {latest_checkpoint}")
    print("⏸️  Training will resume from last checkpoint")
else:
    print("🆕 No checkpoint found - starting fresh training")

print("\n🚀 Starting PaddleOCR training (2-4 hours)...")
print("📊 Checkpoints saved every 5 epochs")
print("⏱️  Use Colab's runtime disconnect prevention if available")

# Run training
!cd /content/egyption_id_ready && python PaddleOCR/tools/train.py -c configs/egyptian_id_rec.yml -o Global.use_gpu=true -o Global.epoch_num=100 -o Global.save_model_dir=/content/egyption_id_ready/output/egyptian_id_rec/

print("\n✅ Training complete!")

## 📊 Part 8: Evaluate Model

In [ ]:
print("📊 Evaluating model...")
!python /content/egyption_id_ready/scripts/evaluate.py --model-path /content/egyption_id_ready/output/egyptian_id_rec/best_accuracy --test-data /content/egyption_id_ready/rec/test.txt
print("\n✅ Evaluation complete!")

## 📦 Part 9: Export to ONNX

In [ ]:
!mkdir -p /content/egyption_id_ready/onnx
!python /content/egyption_id_ready/PaddleOCR/tools/export_model.py -c /content/egyption_id_ready/configs/egyptian_id_rec.yml -o Global.pretrained_model=/content/egyption_id_ready/output/egyptian_id_rec/best_accuracy
!paddle2onnx --model_dir /content/egyption_id_ready/inference/rec --save_file /content/egyption_id_ready/onnx/rec.onnx --opset_version 11
!python -m onnxsim /content/egyption_id_ready/onnx/rec.onnx /content/egyption_id_ready/onnx/rec_sim.onnx
print("\n✅ ONNX export complete!")
!ls -lh /content/egyption_id_ready/onnx/

## 🔌 Part 10: Test Inference

In [ ]:
sys.path.insert(0, '/content/egyption_id_ready')
from src.inference import EgyptianIDOCR

ocr = EgyptianIDOCR(
    det_onnx=str(ROOT / 'model' / 'field_detector.onnx'),
    rec_onnx=str(ROOT / 'onnx' / 'rec_sim.onnx'),
    dict_path=str(ROOT / 'arabic_dict.txt'),
    use_gpu=True,
)
print("✅ OCR loaded!")

test_images = list((ROOT / 'test' / 'images').glob('*.jpg'))
if test_images:
    image = cv2.imread(str(test_images[0]))
    results = ocr.extract(image)
    for field_name, field_data in results.items():
        print(f"{field_name}: {field_data.get('text', 'N/A')}")

### ⚠️ Important: Execution Order

**Before running this section, make sure you have:**

1. ✅ Run **Part 1: Environment Setup** (clones repository)
2. ✅ Run **Part 2: Download Models** (downloads all models)
3. ✅ Run **Part 3: Download Dataset** (downloads dataset)
4. ✅ Run **Part 4: Build Dataset** (processes images)

**Quick Check:** Run this validation cell first:

```python
from pathlib import Path
ROOT = Path('/content/egyption_id_ready')
assert ROOT.exists(), "Repository not found! Run Part 1 first."
assert (ROOT / 'weights').exists(), "Models not found! Run Part 2 first."
assert (ROOT / 'train').exists(), "Dataset not found! Run Part 3 first."
print("✅ All prerequisites met!")
```

## 💾 Part 11: Save to Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/egyption_id_ready/output
!cp -r /content/egyption_id_ready/output/egyptian_id_rec/ /content/drive/MyDrive/egyption_id_ready/output/
!cp -r /content/egyption_id_ready/onnx/ /content/drive/MyDrive/egyption_id_ready/
!cp /content/egyption_id_ready/crops_labeled.csv /content/drive/MyDrive/egyption_id_ready/
print("\n✅ All outputs saved to Google Drive!")

## 📊 Summary

In [ ]:
print("=" * 60)
print("🎉 EGYPTIAN ID OCR - COLAB NOTEBOOK COMPLETE!")
print("=" * 60)
print("\n✅ Accomplished:")
print("   1. Environment setup with GPU")
print("   2. Downloaded YOLO + OCR models")
print("   3. Processed dataset")
print("   4. Labeled crops with OCR")
print("   5. Trained PaddleOCR")
print("   6. Evaluated model")
print("   7. Exported to ONNX")
print("   8. Tested inference")
print("   9. Saved to Google Drive")
print("\n📂 Outputs:")
        print(f"   - Model: /content/egyption_id_ready/output/")
        print(f"   - ONNX: /content/egyption_id_ready/onnx/")
        print(f"   - Drive: /content/drive/MyDrive/egyption_id_ready/")
        print("=" * 60)

In [ ]:
# Final validation and summary
from pathlib import Path
import pandas as pd

print("=" * 60)
print("🎉 FINAL VALIDATION")
print("=" * 60)

ROOT = Path('/content/egyption_id_ready')

# Check all outputs
outputs = {
    'Trained Model': ROOT / 'output' / 'egyptian_id_rec' / 'best_accuracy.pdparams',
    'ONNX Model': ROOT / 'onnx' / 'rec_sim.onnx',
    'Labeled Data': ROOT / 'crops_labeled.csv',
    'Metadata': ROOT / 'crops_metadata_two_stage.csv',
}

print("\n📂 Output Files:")
all_ok = True
for name, path in outputs.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"  ✅ {name}: {size_mb:.2f} MB")
    else:
        print(f"  ❌ {name}: MISSING")
        all_ok = False

# Check Google Drive backup
drive_path = Path('/content/drive/MyDrive/egyption_id_ready')
if drive_path.exists():
    print(f"\n💾 Google Drive Backup:")
    for item in drive_path.rglob('*'):
        if item.is_file():
            print(f"  ✅ {item.relative_to(drive_path)}")

print("\n" + "=" * 60)
if all_ok:
    print("✅ ALL VALIDATIONS PASSED!")
    print("\n🚀 Your Egyptian ID OCR system is ready!")
    print("\n📋 Next Steps:")
    print("   1. Download models from Google Drive")
    print("   2. Deploy API using Docker or locally")
    print("   3. Integrate with your application")
else:
    print("⚠️  Some outputs are missing. Check error messages above.")
print("=" * 60)